# Single File Redundant Averaging

**by Josh Dillon**, last updated September 11, 2026

This notebook applies the smoothed gains from
[calibration_smoothing](https://github.com/HERA-Team/hera_notebook_templates/blob/master/notebooks/phase_II/calibration_smoothing.ipynb)
to a single raw file and coherently averages the calibrated visibilities within redundant baseline
groups, using the per-file `SNAPDecoherence` sidecars from
[file_sky_calibration](https://github.com/HERA-Team/hera_notebook_templates/blob/master/notebooks/phase_II/file_sky_calibration.ipynb)
to correct inter-SNAP cross-correlations for decoherence. It is patterned on H6C's
`file_postprocessing.ipynb` without the abs-cal, diff, delay-filtered, incoherently averaged, and
baseline-selected products, none of which Phase II uses. It also reports the array's remaining
non-redundancy after smoothing, per antenna and per baseline.

Configuration comes from a TOML file (e.g.
`hera_pipelines/pipelines/phase_II/idr1/v1/analysis/phase_II_analysis.toml`) pointed to by the
`TOML_FILE` environment variable; without one, the defaults in the cells below are used. Other
environment variables carry only paths and wrapper-level toggles.

When `SAVE_RESULTS` is `TRUE`, one file is written alongside the input, named from `SUM_FILE` by
replacing `.uvh5`:

* `*.smooth_calibrated.red_avg.uvh5` — smooth-calibrated, decoherence-corrected, redundantly
  averaged visibilities, one per redundant group, keyed by the group's first baseline, with the same
  baselines in every file of the night. **Its `nsamples` are effective numbers of samples, not
  counts.**

A fully-flagged file still produces this file (entirely flagged), so an absent file always means a
failed job.

Here's a set of links to skip to particular figures:

• [Figure 1: Redundant Averaging of the Most Redundant Baseline Groups](#Figure-1:-Redundant-Averaging-of-the-Most-Redundant-Baseline-Groups)

• [Figure 2: Effective Number of Samples as a Function of Baseline](#Figure-2:-Effective-Number-of-Samples-as-a-Function-of-Baseline)

• [Figure 3: Redundant-Baseline chi^2 per Antenna After Smoothing](#Figure-3:-Redundant-Baseline-chi^2-per-Antenna-After-Smoothing)

• [Figure 4: Non-Redundancy of Individual Baselines](#Figure-4:-Non-Redundancy-of-Individual-Baselines)

In [ ]:
import time
tstart = time.time()
!hostname
!date

In [ ]:
import os
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
import h5py
import hdf5plugin  # REQUIRED to have the compression plugins available
import toml
import json
import warnings
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from hera_cal import io, utils, redcal, apply_cal, datacontainer
from hera_qm.metrics_io import read_a_priori_ant_flags
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
%matplotlib inline

## Parse inputs and outputs

To run interactively, provide a sum file path if the environment variable is not set; everything
else has defaults.

In [ ]:
# parse wrapper-level environment variables: paths, plus the save toggle
SUM_FILE = os.environ.get("SUM_FILE", None)
# SUM_FILE = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/2459935/zen.2459935.38700.sum.uvh5'
TOML_FILE = os.environ.get("TOML_FILE", None)
# TOML_FILE = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/src/hera_pipelines/pipelines/phase_II/idr1/v1/analysis/phase_II_analysis.toml'
SAVE_RESULTS = os.environ.get("SAVE_RESULTS", "TRUE").upper() == "TRUE"
# which [WorkFlow] action is running this notebook
ACTION = os.environ.get("ACTION", "FILE_RED_AVG_NOTEBOOK")

# default suffixes for the files this notebook reads or writes
SUM_SUFFIX = 'sum.uvh5'
SMOOTH_CAL_SUFFIX = 'sum.smooth.calfits'
DECOHERENCE_SUFFIX = 'sum.snap_decoherence.h5'
RED_AVG_SUFFIX = 'sum.smooth_calibrated.red_avg.uvh5'

# frequency dividing the low and high bands (inside the a priori flagged FM gap), shared
# across the pipeline via [GLOBAL_OPTS]
BAND_SPLIT_FREQ = 100.0  # in MHz

# default settings, overridden by the TOML's [FILE_RED_AVG_OPTS] section (if given)
INCLUDE_CROSS_POLS = True  # also average en/ne cross-correlations (and cross-polarized autocorrelations)

toml_options = (toml.load(TOML_FILE) if TOML_FILE is not None else {})

# Take the suffixes this notebook reads or writes from [DATA_PRODUCTS], scoped by that
# section's produced_by/consumed_by wiring, and require that wiring to agree exactly with
# the defaults above. That way the arrows drawn on the pipeline flowchart cannot drift from
# what this notebook actually touches: adding an edge there without using the file here (or
# vice versa) fails loudly, in the first cell, rather than silently.
declared_suffixes = {name for name in list(globals()) if name.endswith('_SUFFIX')}
wired_suffixes = set()
for product, spec in toml_options.get('DATA_PRODUCTS', {}).items():
    producers, consumers = (spec.get(key, []) for key in ['produced_by', 'consumed_by'])
    producers = ([producers] if isinstance(producers, str) else producers)
    consumers = ([consumers] if isinstance(consumers, str) else consumers)
    if 'suffix' in spec and (ACTION in producers or ACTION in consumers):
        wired_suffixes.add(f'{product.upper()}_SUFFIX')
        globals()[f'{product.upper()}_SUFFIX'] = spec['suffix']
if toml_options:
    assert wired_suffixes == declared_suffixes, (
        f'[DATA_PRODUCTS] wires {sorted(wired_suffixes)} to {ACTION}, '
        f'but this notebook declares {sorted(declared_suffixes)}.')

for toml_section in ['GLOBAL_OPTS', 'FILE_RED_AVG_OPTS']:
    if toml_section in toml_options:
        print(f'Loading overrides from [{toml_section}] in {TOML_FILE}.')
        for key, val in toml_options[toml_section].items():
            globals()[key.upper()] = val

SMOOTH_CAL_FILE = SUM_FILE.replace(SUM_SUFFIX, SMOOTH_CAL_SUFFIX)
DECOHERENCE_FILE = SUM_FILE.replace(SUM_SUFFIX, DECOHERENCE_SUFFIX)
RED_AVG_FILE = SUM_FILE.replace(SUM_SUFFIX, RED_AVG_SUFFIX)
# the day's final flags, written by calibration_smoothing (the last stage that touches flags)
aposteriori_yaml_file = os.path.join(os.path.dirname(SUM_FILE), SUM_FILE.split('.')[-4] + '_aposteriori_flags.yaml')

for setting in ['SUM_FILE', 'TOML_FILE', 'SAVE_RESULTS', 'ACTION', 'SUM_SUFFIX', 'SMOOTH_CAL_SUFFIX',
                'DECOHERENCE_SUFFIX', 'RED_AVG_SUFFIX', 'BAND_SPLIT_FREQ', 'INCLUDE_CROSS_POLS', 'SMOOTH_CAL_FILE', 'DECOHERENCE_FILE', 'RED_AVG_FILE',
                'aposteriori_yaml_file']:
    print(f'{setting} = {eval(setting)}')

## Load calibration, decoherence, and data

The smoothed gains carry each file's fitted decoherence staircase, exactly like the per-file
`sky.calfits`. Identity relabelings recorded in the calibration's `RELABELS` keyword are applied to
the raw data so that its antenna numbers agree with the gains, the antenna positions, and the
sidecar's antenna → SNAP map. Antennas whose SNAP has no decoherence measurement in this file are
flagged, since their inter-SNAP baselines cannot be corrected.

In [ ]:
hc = io.HERACal(SMOOTH_CAL_FILE)
gains, cal_flags, _, _ = hc.read()
relabels = {int(labeled): int(true_ant)
            for labeled, true_ant in json.loads(hc.extra_keywords.get('RELABELS', '{}')).items()}
sd = io.SNAPDecoherence.read(DECOHERENCE_FILE)
ALL_FLAGGED = bool(np.all([cal_flags[ant] for ant in cal_flags]))

In [ ]:
pols = (['ee', 'nn', 'en', 'ne'] if INCLUDE_CROSS_POLS else ['ee', 'nn'])
hd = io.HERADataFastReader(SUM_FILE)
data, _, _ = hd.read(pols=pols, read_flags=False, read_nsamples=False)  # raw data: no flags, nsamples all 1

assert np.allclose(sd.times, hd.times, rtol=0, atol=1e-8), f'{DECOHERENCE_FILE} does not cover the times in {SUM_FILE}.'
assert all(gains[ant].shape == (len(hd.times), len(hd.freqs)) for ant in gains), f'{SMOOTH_CAL_FILE} does not match the shape of {SUM_FILE}.'

In [ ]:
if len(relabels) > 0:
    def fix_key(bl):
        return (relabels.get(bl[0], bl[0]), relabels.get(bl[1], bl[1]), bl[2])
    data = datacontainer.DataContainer({fix_key(bl): data[bl] for bl in data})
    print('Identity repairs applied: ' + '; '.join(f'visibilities labeled {labeled} reassigned to antenna {true_ant}'
                                                  for labeled, true_ant in sorted(relabels.items())) + '.')

In [ ]:
uncorrectable = sorted(ant for ant in gains if not np.all(cal_flags[ant])
                       and sd.ant_to_SNAP_dict.get(ant[0]) not in sd.decoherence)
for ant in uncorrectable:
    cal_flags[ant][:] = True
if len(uncorrectable) > 0:
    print(f'Flagging {len(uncorrectable)} antpol(s) without a decoherence measurement in this file: {uncorrectable}')
ALL_FLAGGED = ALL_FLAGGED or bool(np.all([cal_flags[ant] for ant in cal_flags]))

In [ ]:
# every redundant group the data could populate, so that every file of the night writes the same
# baselines; groups whose antennas are all flagged for the whole night (per the a posteriori yaml)
# are dropped
reds = redcal.get_reds(hd.data_antpos, pols=pols, include_autos=True)
day_ex_ants = set(read_a_priori_ant_flags(aposteriori_yaml_file))
reds = [red for red in reds if any(not any(ant in day_ex_ants for ant in utils.split_bl(bl)) for bl in red)]
print(f'Averaging {len(reds)} redundant groups, excluding those made up entirely of the {len(day_ex_ants)} antpols flagged all night.')

In [ ]:
# for calibrating individual members of a group, the way calibrate_and_red_avg does it internally
clean_gains = sd.correct_gains({ant: gains[ant] for ant in gains if ant[0] in sd.ant_to_SNAP_dict})
unmeasured = {SNAP: np.repeat(np.isnan(p), sd.block_freqs.shape[1], axis=1) for SNAP, p in sd.decoherence.items()}
dt = np.median(np.diff(hd.times)) * 24 * 3600
df = np.median(np.diff(hd.freqs))

def calibrated_members(red):
    '''Calibrated, decoherence-corrected members of a redundant group, with flagged cells set to nan.'''
    members = [bl for bl in red if bl in data
               and all(ant in clean_gains and not np.all(cal_flags[ant]) for ant in utils.split_bl(bl))]
    if len(members) == 0:
        return {}
    group = datacontainer.DataContainer({bl: data[bl].copy() for bl in members})
    group_flags = {}
    for bl in members:
        ant_i, ant_j = utils.split_bl(bl)
        group_flags[bl] = cal_flags[ant_i] | cal_flags[ant_j]
        if sd.ant_to_SNAP_dict[bl[0]] != sd.ant_to_SNAP_dict[bl[1]]:  # inter-SNAP: unmeasured blocks cannot be corrected
            group_flags[bl] = group_flags[bl] | unmeasured[sd.ant_to_SNAP_dict[bl[0]]] | unmeasured[sd.ant_to_SNAP_dict[bl[1]]]
    apply_cal.calibrate_in_place(group, clean_gains)
    sd.correct_in_place(group, data_flags=datacontainer.DataContainer(group_flags))
    return {bl: np.where(group_flags[bl], np.nan, group[bl]) for bl in members}

## Calibrate and redundantly average

`apply_cal.calibrate_and_red_avg` works one redundant group at a time:

* the fitted decoherence staircase is cleaned out of the gains (`SNAPDecoherence.correct_gains`), as
  autocorrelations and intra-SNAP baselines require;
* cross-correlations are inverse-variance weighted with $\sigma^2 = A_i A_j / (\Delta t \, \Delta
  \nu)$ from the calibrated autocorrelations, inflated by $1 / ((1 - p_i)(1 - p_j))^2$ for
  inter-SNAP baselines;
* inter-SNAP cross-correlations are divided by $(1 - p_i)(1 - p_j)$, with blocks where either SNAP's
  decoherence is unmeasured flagged;
* the returned `nsamples` are *effective*, $\bar{A}_i \bar{A}_j \sum w / (\Delta t \, \Delta \nu)$,
  so that the standard noise prediction from the averaged autocorrelations is exact for every
  averaged product;
* each member's DoF-normalized scatter about its group mean accumulates into a redundant-baseline
  $\chi^2$ per antenna and per baseline. Since the gains were fit to the sky model rather than to
  redundancy, this measures intrinsic non-redundancy plus calibration error in units of thermal
  noise and is not expected to be ~1.

Antennas flagged upstream are excluded from the averages, the effective numbers of samples, and the $\chi^2$.

In [ ]:
red_avg_data, red_avg_flags, red_avg_nsamples, red_avg_meta = apply_cal.calibrate_and_red_avg(
    data, gains, reds, ant_flags=cal_flags, snap_decoherence=sd, dt=dt, df=df)
ALL_FLAGGED = ALL_FLAGGED or bool(np.all([red_avg_flags[bl] for bl in red_avg_flags]))
for pol in ['Jee', 'Jnn']:
    if np.any(np.isfinite(red_avg_meta['total_chisq'].get(pol, np.nan))):
        print(f'Median unflagged redundant-baseline chi^2 / DoF for {pol}: {np.nanmedian(red_avg_meta["total_chisq"][pol]):.3f}')

In [ ]:
def plot_red_avg_vis(pols_to_plot=['ee', 'nn']):
    if ALL_FLAGGED:
        print('All integrations are flagged. Nothing to plot.')
        return

    fig, axes = plt.subplots(2, 2, figsize=(14, 6), dpi=150, sharex='col', sharey='row', gridspec_kw={'hspace': 0, 'wspace': 0})
    for i, pol in enumerate(pols_to_plot):
        # the cross-correlation group with the most effective samples
        candidates = [red for red in reds if red[0][2] == pol and red[0][0] != red[0][1] and red[0] in red_avg_data]
        if len(candidates) == 0:
            continue
        red = max(candidates, key=lambda red: np.median(red_avg_nsamples[red[0]]))
        tind = np.argmin(np.all(red_avg_flags[red[0]], axis=1))  # first integration not entirely flagged
        members = calibrated_members(red)
        for bl in members:
            axes[0, i].plot(hd.freqs / 1e6, np.angle(members[bl][tind]), alpha=.5, lw=.5)
            axes[1, i].semilogy(hd.freqs / 1e6, np.abs(members[bl][tind]), alpha=.5, lw=.5)

        to_plot = np.where(red_avg_flags[red[0]][tind], np.nan, red_avg_data[red[0]][tind])
        n_members = sum(not np.all(np.isnan(members[bl][tind])) for bl in members)
        med_nsamples = np.nanmedian(np.where(red_avg_flags[red[0]][tind], np.nan, red_avg_nsamples[red[0]][tind]))
        axes[0, i].plot(hd.freqs / 1e6, np.angle(to_plot), lw=1, c='k')
        axes[1, i].semilogy(hd.freqs / 1e6, np.abs(to_plot), lw=1, c='k',
                            label=f'Baseline Group {(int(red[0][0]), int(red[0][1]), red[0][2])}:\n'
                                  f'{n_members} baselines, {med_nsamples:.1f} effective samples')
        axes[1, i].set_xlabel('Frequency (MHz)')
        axes[1, i].legend(loc='upper right')
    axes[0, 0].set_ylabel('Visibility Phase (radians)')
    axes[1, 0].set_ylabel('Visibility Amplitude (Jy)')
    plt.tight_layout()

# *Figure 1: Redundant Averaging of the Most Redundant Baseline Groups*

The calibrated, decoherence-corrected members (thin colored lines) and their inverse-variance
weighted average (black) for the cross-correlation group with the most effective samples in each
polarization, at the first unflagged integration. Top: phases; bottom: amplitudes; left: ee; right:
nn.

In [ ]:
plot_red_avg_vis()

In [ ]:
def plot_red_avg_nsamples():
    if ALL_FLAGGED:
        print('All integrations are flagged. Nothing to plot.')
        return

    fig, axes = plt.subplots(2, 1, figsize=(14, 7), dpi=150, sharex=True, gridspec_kw={'hspace': 0})
    med_nsamples = {red[0]: np.nanmedian(np.where(red_avg_flags[red[0]], np.nan, red_avg_nsamples[red[0]]))
                    for red in reds if red[0] in red_avg_data and not np.all(red_avg_flags[red[0]])}
    for ax, pol in zip(axes, ['ee', 'nn']):
        bls_here = [bl for bl in med_nsamples if bl[2] == pol]
        if len(bls_here) > 0:
            blvecs = np.array([hd.antpos[bl[1]] - hd.antpos[bl[0]] for bl in bls_here])
            sca = ax.scatter(blvecs[:, 0], blvecs[:, 1], s=0)
            sca = ax.scatter(blvecs[:, 0], blvecs[:, 1], s=(100 * (600 / np.diff(ax.get_xlim())[0])**2), ec='k', linewidths=.5,
                             c=[med_nsamples[bl] for bl in bls_here], cmap='turbo',
                             norm=matplotlib.colors.LogNorm(vmin=1, vmax=max(med_nsamples.values())))
            ax.axis('equal')
        ax.set_xlabel('EW Baseline Vector (m)')
        ax.set_ylabel('NS Baseline Vector (m)')
        ax.text(.98, .94, f'{pol}-polarized', transform=ax.transAxes, va='top', ha='right', bbox=dict(facecolor='w', alpha=0.5))

    plt.tight_layout()
    fig.colorbar(sca, ax=axes, pad=.02, label='Median Effective Number of Samples')

# *Figure 2: Effective Number of Samples as a Function of Baseline*

The median effective number of samples (over unflagged times and channels) of each redundantly
averaged cross-correlation, by baseline vector. The split of the HERA core produces highly-sampled
intra-sector baselines interspersed with less-sampled inter-sector ones off the main grid. Effective
samples exceed the member count where quiet antennas dominate the average and fall short of it where
inter-SNAP baselines were corrected for decoherence.

In [ ]:
plot_red_avg_nsamples()

## Examine non-redundancy after smoothing

In [ ]:
def chisq_array_plot(chisq_per_ant, statistic, vmax=None):
    '''Array plot of a per-antenna chi^2 statistic, averaged over its unflagged times and channels.
    vmax=None picks a robust scale from the data.'''
    if ALL_FLAGGED:
        print('All integrations are flagged. Nothing to plot.')
        return
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        avgs = {ant: np.nanmean(np.where(cal_flags[ant], np.nan, cspa)) for ant, cspa in chisq_per_ant.items()}
    if vmax is None:
        vmax = max(2, np.nanpercentile([m for m in avgs.values() if np.isfinite(m)], 90))

    def _chisq_subplot(antnums, size=250):
        fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)
        for ax, pol in zip(axes, ['Jee', 'Jnn']):
            # invisible scatter of every antenna position so flagged (chi^2-less) antennas
            # still fall inside the axes limits
            ax.scatter([hd.antpos[antnum][0] for antnum in antnums],
                       [hd.antpos[antnum][1] for antnum in antnums], s=size, facecolors='none', edgecolors='none')
            finite = [(antnum, pol) for antnum in antnums if np.isfinite(avgs.get((antnum, pol), np.nan))]
            for antnum in antnums:
                ax.text(hd.antpos[antnum][0], hd.antpos[antnum][1], antnum, va='center', ha='center', fontsize=8,
                        c=('w' if (antnum, pol) in finite else 'r'))
            ax.axis('equal')
            ax.set_xlabel('East-West Position (meters)')
            ax.set_ylabel('North-South Position (meters)')
            ax.set_title(f'{pol[1:]}-pol Mean {statistic} / Antenna')
            if len(finite) == 0:
                continue
            scatter = ax.scatter([hd.antpos[ant[0]][0] for ant in finite], [hd.antpos[ant[0]][1] for ant in finite],
                                 s=size, c=[avgs[ant] for ant in finite], lw=.25, edgecolors='none', zorder=-1,
                                 norm=matplotlib.colors.LogNorm(vmin=1, vmax=vmax))
            plt.colorbar(scatter, ax=ax, extend='both')
        plt.tight_layout()

    _chisq_subplot([antnum for antnum in hd.data_ants if antnum < 320])
    outriggers = [antnum for antnum in hd.data_ants if antnum >= 320
                  and any(np.isfinite(avgs.get((antnum, pol), np.nan)) for pol in ['Jee', 'Jnn'])]
    if len(outriggers) > 0:
        _chisq_subplot(outriggers, size=400)

# *Figure 3: Redundant-Baseline chi^2 per Antenna After Smoothing*

The mean over unflagged times and channels of each antenna's DoF-normalized scatter about its
redundant-group means. Unlike the per-file version in `file_sky_calibration`, this uses the smoothed
gains, so it also includes any real per-file gain structure that smoothing removed. It is not
expected to be ~1; what matters is relative structure across the array. Antennas numbered in red are
flagged.

In [ ]:
chisq_array_plot(red_avg_meta['chisq_per_ant'], 'Redundant-Baseline $\\chi^2$')

In [ ]:
def plot_baseline_nonredundancy(n_worst=10):
    if ALL_FLAGGED:
        print('All integrations are flagged. Nothing to plot.')
        return
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        bl_chisq = {bl: np.nanmean(c) for bl, c in red_avg_meta['chisq_per_bl'].items() if np.any(np.isfinite(c))}

    # each baseline's chi^2 relative to the median of its redundant group (groups of one carry no information)
    ratios, group_sizes = {}, {}
    for red in reds:
        members = [bl for bl in red if bl in bl_chisq]
        if len(members) > 1:
            group_median = np.median([bl_chisq[bl] for bl in members])
            for bl in members:
                ratios[bl] = bl_chisq[bl] / group_median
                group_sizes[bl] = len(members)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)
    for pol, color in [('ee', 'C0'), ('nn', 'C1')]:
        bls_here = [bl for bl in ratios if bl[2] == pol]
        if len(bls_here) == 0:
            continue
        lengths = [np.linalg.norm(hd.antpos[bl[1]] - hd.antpos[bl[0]]) for bl in bls_here]
        axes[0].scatter(lengths, [bl_chisq[bl] for bl in bls_here], s=4, alpha=.3, color=color, label=f'{pol}-polarized')
        axes[1].hist([ratios[bl] for bl in bls_here], bins=np.logspace(-1.5, 2.5, 81), alpha=.5, color=color, label=f'{pol}-polarized')
    axes[0].set_yscale('log')
    axes[0].set_xlabel('Baseline Length (m)')
    axes[0].set_ylabel('Mean Unflagged Redundant-Baseline $\\chi^2$ / DoF')
    axes[0].legend()
    axes[1].set_xscale('log')
    axes[1].set_yscale('log')
    axes[1].set_xlabel('$\\chi^2$ / DoF Relative to Redundant Group Median')
    axes[1].set_ylabel('Number of Baselines')
    axes[1].legend()
    plt.tight_layout()

    print('Baselines most discrepant with their redundant groups (chi^2 / DoF relative to the group median):')
    for bl in sorted(ratios, key=ratios.get, reverse=True)[:n_worst]:
        print(f'    {(int(bl[0]), int(bl[1]), bl[2])}: {ratios[bl]:.1f}x the median of its group of {group_sizes[bl]} '
              f'(chi^2 / DoF = {bl_chisq[bl]:.2f})')

# *Figure 4: Non-Redundancy of Individual Baselines*

Left: each averaged baseline's mean unflagged redundant-baseline $\chi^2$ / DoF against its length.
Right: the same $\chi^2$ relative to the median of its redundant group, isolating baselines that
disagree with their group-mates beyond the non-redundancy they all share; the most discrepant are
listed above the figure. Since a discrepant member also pulls the group mean toward itself, its
$\chi^2$ can exceed its group-mates' by at most ~(N−1)² in an N-member group, so this is only
informative for groups of several baselines. No baselines are excluded on this basis (yet).

In [ ]:
plot_baseline_nonredundancy()

## Save redundantly averaged visibilities

Every group is written, keyed by its first baseline; groups with no usable members get fully-flagged
placeholders so that every file of the night has the same baselines. The `nsamples` written are
effective numbers of samples.

In [ ]:
add_to_history = ('Produced by file_redundant_averaging notebook with the following environment:\n' + '=' * 65 + '\n'
                  + os.popen('conda env export').read() + '=' * 65
                  + '\nnsamples are effective numbers of samples (see hera_cal.apply_cal.calibrate_and_red_avg), not counts.')

if SAVE_RESULTS:
    shape = (len(hd.times), len(hd.freqs))
    out_data = {red[0]: (red_avg_data[red[0]] if red[0] in red_avg_data else np.zeros(shape, dtype=complex)) for red in reds}
    out_flags = {red[0]: (red_avg_flags[red[0]] if red[0] in red_avg_flags else np.ones(shape, dtype=bool)) for red in reds}
    out_nsamples = {red[0]: (red_avg_nsamples[red[0]] if red[0] in red_avg_nsamples else np.zeros(shape)) for red in reds}

    hd_out = io.HERAData(SUM_FILE)
    antpairs = sorted(set(red[0][:2] for red in reds))
    hd_out.read(bls=antpairs, polarizations=pols)
    hd_out.empty_arrays()
    hd_out.update(data=out_data, flags=out_flags, nsamples=out_nsamples)
    # the calibration carries the sky model's units and polarization convention (placeholders included),
    # which every file of the night must share for the corner turn to concatenate them
    assert hc.gain_scale is not None, f'{SMOOTH_CAL_FILE} records no gain_scale, so {RED_AVG_FILE} would not match the rest of the night.'
    hd_out.vis_units = hc.gain_scale
    hd_out.pol_convention = hc.pol_convention
    if len(relabels) > 0:
        hd_out.extra_keywords['RELABELS'] = hc.extra_keywords['RELABELS']
    hd_out.history += add_to_history
    print(f'Now writing redundantly averaged calibrated visibilities to {RED_AVG_FILE}')
    hd_out.write_uvh5(RED_AVG_FILE, clobber=True, fix_autos=True)

## Metadata

In [ ]:
for repo in ['hera_cal', 'hera_qm', 'hera_filters', 'hera_notebook_templates', 'pyuvdata']:
    exec(f'from {repo} import __version__')
    print(f'{repo}: {__version__}')

In [ ]:
print(f'Finished execution in {(time.time() - tstart) / 60:.2f} minutes.')